# Notebook 01 — Data Extraction

**Project:** Why Startups Fail — VC Investment Pattern Analysis  
**Team:** Section C, Group 17  
**Dataset:** Crunchbase VC Investments Dataset  
**Source:** Kaggle (raw transactional startup investment records)  

---
**Objective:** Load the raw dataset, perform an initial structural inspection, validate row/column counts, and document the extraction metadata before any transformation is applied.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

Libraries loaded successfully.
Pandas version: 2.3.3
NumPy version: 2.3.5


In [2]:
RAW_DATA_PATH = '/content/investments_VC (1).csv'


In [5]:
df_raw = pd.read_csv(RAW_DATA_PATH, encoding='latin-1', low_memory=False)

print('Dataset loaded successfully.')
print(f'Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')

FileNotFoundError: [Errno 2] No such file or directory: '/content/investments_VC (1).csv'

In [6]:
# Display first 5 rows
df_raw.head()

NameError: name 'df_raw' is not defined

In [14]:
# Column names and dtypes
print('Column Names and Data Types:')
print('=' * 50)
for col in df_raw.columns:
    print(f'  {col.strip():<35} {str(df_raw[col].dtype)}')

Column Names and Data Types:
  permalink                           object
  name                                object
  homepage_url                        object
  category_list                       object
  market                              object
  funding_total_usd                   object
  status                              object
  country_code                        object
  state_code                          object
  region                              object
  city                                object
  funding_rounds                      float64
  founded_at                          object
  founded_month                       object
  founded_quarter                     object
  founded_year                        float64
  first_funding_at                    object
  last_funding_at                     object
  seed                                float64
  venture                             float64
  equity_crowdfunding                 float64
  undisclosed        

In [15]:
# Count missing values per column
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print('Missing Value Report:')
print(missing_report[missing_report['Missing Count'] > 0].to_string())

Missing Value Report:
                      Missing Count  Missing %
state_code                    24133      44.45
founded_quarter               15812      29.12
founded_month                 15812      29.12
founded_year                  15812      29.12
founded_at                    15740      28.99
city                          10972      20.21
region                        10129      18.66
country_code                  10129      18.66
 market                        8824      16.25
category_list                  8817      16.24
homepage_url                   8305      15.30
status                         6170      11.36
name                           4857       8.95
 funding_total_usd             4856       8.94
permalink                      4856       8.94
funding_rounds                 4856       8.94
first_funding_at               4856       8.94
last_funding_at                4856       8.94
seed                           4856       8.94
venture                        4856   

In [16]:
# Check cardinality of key categorical fields
categorical_cols = ['status', 'country_code', 'state_code', 'market']
for col in categorical_cols:
    col_clean = col.strip()
    # Handle possible whitespace in column names
    matching = [c for c in df_raw.columns if c.strip() == col_clean]
    if matching:
        vals = df_raw[matching[0]].value_counts(dropna=False)
        print(f'\n--- {col_clean} ({len(vals)} unique values) ---')
        print(vals.head(15).to_string())


--- status (4 unique values) ---
status
operating    41829
NaN           6170
acquired      3692
closed        2603

--- country_code (116 unique values) ---
country_code
USA    28793
NaN    10129
GBR     2642
CAN     1405
CHN     1239
DEU      968
FRA      866
IND      849
ISR      682
ESP      549
RUS      368
SWE      315
AUS      314
ITA      308
NLD      307

--- state_code (62 unique values) ---
state_code
NaN    24133
CA      9917
NY      2914
MA      1969
TX      1466
WA       974
FL       963
IL       827
PA       792
CO       723
ON       653
NJ       579
VA       553
GA       541
OH       532

--- market (754 unique values) ---
 market 
NaN                      8824
 Software                4620
 Biotechnology           3688
 Mobile                  1983
 E-Commerce              1805
 Curated Web             1655
 Enterprise Software     1280
 Health Care             1207
 Clean Technology        1200
 Games                   1182
 Hardware + Software     1081
 Advertising 

In [17]:
import datetime

metadata = {
    'Dataset Name': 'investments_VC.csv',
    'Source': 'Crunchbase via Kaggle (raw startup investment records)',
    'Extraction Date': str(datetime.date.today()),
    'Total Rows': df_raw.shape[0],
    'Total Columns': df_raw.shape[1],
    'File Size (MB)': round(os.path.getsize(RAW_DATA_PATH) / (1024*1024), 2),
    'Date Range (Founded)': f"{df_raw['founded_year'].min()} – {df_raw['founded_year'].max()}",
    'Countries': df_raw['country_code'].nunique(),
    'Startup Statuses': df_raw['status'].unique().tolist()
}

print('=== EXTRACTION METADATA ===')
for k, v in metadata.items():
    print(f'{k:<30}: {v}')

=== EXTRACTION METADATA ===
Dataset Name                  : investments_VC.csv
Source                        : Crunchbase via Kaggle (raw startup investment records)
Extraction Date               : 2026-04-21
Total Rows                    : 54294
Total Columns                 : 39
File Size (MB)                : 11.95
Date Range (Founded)          : 1902.0 – 2014.0
Countries                     : 115
Startup Statuses              : ['acquired', 'operating', nan, 'closed']


## 1.7 — Extraction Notes

| Issue Observed | Column | Action Planned |
|---|---|---|
| Whitespace in column names | `market`, `funding_total_usd` | Strip in cleaning |
| Funding formatted as string with commas | `funding_total_usd` | Convert to numeric |
| Mixed date formats | `founded_at`, `first_funding_at`, `last_funding_at` | Parse to datetime |
| High missing rate | `state_code`, `region`, `city` | Impute or drop |
| Pipe-separated values | `category_list` | Parse in feature engineering |
| Startup name anomalies | `name` | Keep as-is, flag duplicates |

All transformations will be applied in **Notebook 02 — Cleaning**.